# Chapter 01-05 · pandas II: grouping, joining, and time

**Label:** Optional  |  **Time:** ~55 minutes  |  **Difficulty:** moderate - two of the labs are bugs you will meet in real work

**Prerequisites:** 01-04. You should be able to load a file, read `.info()`, and select with
`.loc`.

**Position in the learning path:** module 01, chapter 5 of 6. Before: **01-04**. After:
**01-06** (plotting and seeds), which closes the module.

---

## Why this matters

Three operations turn a table into an answer, and this chapter is all three:

- **`groupby`** - because "compute this separately for each group" *is* error analysis. Every time
  a later chapter asks where a model fails - by weather, by hour, by customer type - the answer is
  a `groupby`.
- **joins** - because data arrives in several tables, and because a join that silently changes the
  row count is how duplicated observations get into a training set. That is leakage, and it is the
  subject of 04-05.
- **time** - because a missing day in a time series shifts every lag feature after it, and module
  09 is built on lags.

Two of those sentences describe bugs that produce a *better-looking* result. Those are the two
failure labs.

## What you will be able to do

By the end of this chapter you can:

1. **Use** `groupby` with one or several keys, and with several aggregations at once.
2. **Choose** the right join type and **check the row count** before and after, every time.
3. **Diagnose** a join that multiplied rows, and say why that inflates a model's score.
4. **Work** with timestamps: parse them, extract parts, index by them, and resample.
5. **Explain** why grouping by `.dt.date` and resampling by day give different answers, and when
   that difference is a bug.

## Warm-up: retrieve, do not reread

From memory:

1. What is the first line you run after loading a file, and what three things do you read from it?
2. Why is `max()` on a text column silently wrong?
3. Why did the edit through a filtered subset vanish?
4. What does `axis=n` do?

<br>

*Answers: (1) `df.info()` - row count, non-null counts, dtypes. (2) it compares alphabetically, so
`'9,0'` beats `'31,7'`. (3) boolean indexing returns a copy, so the assignment modified the copy.
(4) names the axis that disappears.*

## The situation

Maria's stand now has three docking stations, and the data lives in two tables - which is how data
always lives.

- `rentals`: one row per station per day, with a count.
- `stations`: one row per station, with its site and capacity.

The questions are ordinary: *how busy is each site? which station is busiest per unit of capacity?
what does the week look like?* Answering them needs a group, a join, and a date index.

**The question this chapter answers:** how do you combine and summarise tables without silently
changing what the data says?

In [ ]:
import numpy as np
import pandas as pd

rentals = pd.DataFrame({
    "station_id": ["a", "b", "c", "a", "b", "c", "a", "b", "c", "a", "b", "c"],
    "day": ["2024-03-01"] * 3 + ["2024-03-02"] * 3 + ["2024-03-04"] * 3 + ["2024-03-05"] * 3,
    "count": [120, 80, 45, 150, 95, 60, 90, 70, 30, 200, 110, 75],
})
stations = pd.DataFrame({
    "station_id": ["a", "b", "c"],
    "site": ["north", "north", "south"],
    "capacity": [40, 25, 15],
})

print("rentals :", rentals.shape, " stations:", stations.shape)
print("days present:", sorted(rentals['day'].unique()), "  <- note what is missing")
rentals.head(4)

March 3rd is not there. Hold that thought - it is the second failure lab.

## groupby: split, apply, combine

`groupby` does three things, always in this order: **split** the rows into groups by a key,
**apply** a function to each group, **combine** the results into one object.

You wrote this yourself in 01-02 as `setdefault(key, []).append(row)` followed by a comprehension.
pandas does the same thing in compiled code, with a much richer set of things you can apply.

In [ ]:
print("total per station:")
print(rentals.groupby("station_id")["count"].sum())
print()
print("several statistics at once:")
print(rentals.groupby("station_id")["count"].agg(["count", "mean", "min", "max"]).round(1))

Read `.groupby("station_id")["count"].sum()` as a sentence: *split by station, take the count
column, add it up*.

`.agg([...])` applies several functions and returns a DataFrame - one row per group, one column
per statistic. This is the shape you want for error analysis: not one number, but a small table
showing where a model is good and where it is not. You saw a first version of it in 00-01.

Two variants worth having now:

In [ ]:
# Group by more than one key.
print(rentals.merge(stations, on="station_id")
      .groupby(["site", "station_id"])["count"].sum())
print()
# Name the outputs, which is much easier to read than a column of tuples.
print(rentals.groupby("station_id").agg(days=("count", "size"),
                                        busiest_day=("count", "max"),
                                        average=("count", "mean")).round(1))

The named form - `new_name=(column, function)` - is worth adopting as a default. The alternative
produces multi-level column names that are awkward to select from and awkward to read.

One more, which is the one people miss:

In [ ]:
# transform gives a result the SAME LENGTH as the input, aligned back to the rows.
rentals_with_share = rentals.assign(
    station_total=lambda d: d.groupby("station_id")["count"].transform("sum"),
    share_of_station=lambda d: (d["count"] / d.groupby("station_id")["count"].transform("sum")).round(3),
)
print(rentals_with_share.head(6))

`agg` collapses each group to one row. **`transform` keeps every row and broadcasts the group's
value back onto it**, which is how you add a column like "this day's share of the station's
total", or "how far above this station's average was today".

That distinction - collapse or broadcast - is the whole of `groupby`. Once you have it, `.apply`
is just "do something arbitrary per group and let pandas work out which of the two you meant",
which is why `.apply` is slower and harder to predict; prefer `agg` or `transform` when you can
name what you want.

## Joining, and the number that must not change

`stations` holds the attributes; `rentals` holds the observations. Bringing them together is a
**join**, and the only question that matters is *how many rows come out*.

### Predict before running

`rentals` has 12 rows. `stations` has 3, one per station.

1. How many rows should a left join produce?
2. What would it mean if you got 24?

In [ ]:
joined = rentals.merge(stations, on="station_id", how="left")

print("rows in:", len(rentals), " rows out:", len(joined))
print(joined.head(3))

Twelve in, twelve out. Each rental row found exactly one station and gained two columns.

The four join types, and what each is *for*:

| `how=` | Keeps | Use when |
|---|---|---|
| `"left"` | every row of the left frame | Enriching observations with attributes - the default for this kind of work |
| `"inner"` | only rows that match on both sides | You genuinely want to drop unmatched rows - and you should say so out loud, because it deletes data silently |
| `"outer"` | everything from both | Reconciling two sources; finding what is missing where |
| `"right"` | every row of the right frame | Rare - usually a left join written backwards |

pandas gives you two tools that turn "count the rows" from a discipline into a guarantee:

In [ ]:
checked = rentals.merge(stations, on="station_id", how="left",
                        validate="many_to_one",     # each rental matches at most ONE station
                        indicator=True)             # add a column saying where each row came from

print("rows:", len(checked))
print(checked["_merge"].value_counts().to_string())

**`validate="many_to_one"`** states the relationship you believe in - many rentals, one station -
and pandas raises if the right-hand keys are not unique. It converts a silent multiplication into
an error at the line that caused it. Use it on every join; the vocabulary is `one_to_one`,
`one_to_many`, `many_to_one`, `many_to_many`.

**`indicator=True`** adds a `_merge` column saying whether each row matched both sides or only
one. Counting it tells you immediately how many rows found no partner - the thing a left join
hides by filling with `NaN`.

---

## Failure lab 1: the join that multiplied the data

Station `b` was relocated last year, so the reference table has two rows for it - the old site and
the new one. Nobody removed the old row, because nobody knew it mattered.

**Predict before running:** `rentals` has 12 rows and the lookup now has 4. How many rows come out
of the left join, and what happens to the total rental count?

In [ ]:
stations_messy = pd.concat([stations,
                            pd.DataFrame({"station_id": ["b"], "site": ["east"], "capacity": [25]})],
                           ignore_index=True)
print(stations_messy)

inflated = rentals.merge(stations_messy, on="station_id", how="left")
print("\nrows in:", len(rentals), " rows out:", len(inflated))
print("true total rentals    :", rentals['count'].sum())
print("total after the join  :", inflated['count'].sum())

### Diagnosis

**Twelve rows in, sixteen out.** Every one of station b's four rental days matched *both* lookup
rows, so each was duplicated. The total rental count rose from 1125 to **1480**, an invented 32%.

No error. No warning. A DataFrame that looks entirely normal.

**Why this specific bug is so expensive in machine learning**, rather than merely wrong:

Those duplicated rows are not just extra weight in a sum. They are **the same observation appearing
twice**. Split that dataset randomly into train and test, and copies of the same day land on both
sides. The model sees a row in training and is then tested on its identical twin, recognises it
perfectly, and scores far better than it will in production.

That is **duplicate leakage** - one of the four kinds in 04-05 - and it is the one most often
created by a join rather than by a modelling decision. A model that mysteriously scores well and
disappoints in production has this in its history more often than any other single cause.

**The three defences, in increasing order of strength:**

In [ ]:
# 1. Count. One line, catches everything, costs nothing.
before, after = len(rentals), len(rentals.merge(stations_messy, on="station_id", how="left"))
print(f"1. counted: {before} -> {after}", "  MISMATCH" if before != after else "  ok")

# 2. Validate. Turns the silent multiplication into an exception at the guilty line.
try:
    rentals.merge(stations_messy, on="station_id", how="left", validate="many_to_one")
except Exception as exc:
    print("2. validate raised:", type(exc).__name__, "-", exc)

# 3. Check the key BEFORE joining, so you find out what is wrong rather than that something is.
duplicated_keys = stations_messy[stations_messy["station_id"].duplicated(keep=False)]
print("\n3. duplicate keys in the lookup:")
print(duplicated_keys)

The third is the one that actually helps you, because it names the row. `validate` tells you the
join is wrong; `.duplicated(keep=False)` shows you *which station and why*, and then a human can
decide whether the right answer is to drop the old row, keep the most recent, or discover that
station b genuinely has two sites and the data model is wrong.

**The habit to leave this chapter with:** print the row count before and after every join, and
pass `validate=`. Together they are two lines and they close off an entire category of silent
disaster.

---

## Time: parsing, extracting, indexing, resampling

Every column that holds a date starts life as text (01-04). Converting it is the first step, and
everything useful follows from it.

In [ ]:
daily = rentals.assign(day=lambda d: pd.to_datetime(d["day"]))
print(daily.dtypes.to_string())
print()
print("year      :", daily["day"].dt.year.unique())
print("weekday   :", daily["day"].dt.day_name().unique().tolist())
print("is weekend:", daily["day"].dt.dayofweek.isin([5, 6]).unique())

The `.dt` accessor is to timestamps what `.str` is to text: a namespace of everything you might
want to extract. `year`, `month`, `day`, `hour`, `dayofweek`, `day_name()`, `quarter`,
`is_month_end` - and every one of them is a candidate feature for a time-aware model (09-02).

Now the more important move: making time the **index**.

In [ ]:
per_day = daily.groupby("day")["count"].sum()
print(per_day)
print()
print("slice by date range:")
print(per_day.loc["2024-03-02":"2024-03-04"])

With a `DatetimeIndex`, `.loc` takes date strings and slices are inclusive of the end - which is
the behaviour anybody would want when they write "from the 2nd to the 4th". This is the reason
`.loc` slices include their endpoint, mentioned in 01-04.

### Predict before running

`per_day` has four entries: the 1st, 2nd, 4th and 5th of March. The 3rd is missing.

1. What does `per_day.resample("D").sum()` give - four rows or five?
2. What does `daily.groupby(daily["day"].dt.date)["count"].sum()` give?
3. Which of the two would you want if you were about to compute "yesterday's rentals" as a feature?

In [ ]:
resampled = per_day.resample("D").sum()
grouped = daily.groupby(daily["day"].dt.date)["count"].sum()

print("resample('D'):")
print(resampled)
print("\ngroupby(.dt.date):")
print(grouped)

## Failure lab 2: the day that was not there

**`resample("D")` returns five rows, including March 3rd with a value of 0. `groupby(.dt.date)`
returns four rows and March 3rd simply does not exist.**

Both are defensible. Only one of them is safe.

**Why it matters.** Suppose you build the feature every forecasting model starts with - *yesterday's
rentals*. With the grouped version, the row for March 4th takes "the previous row", which is March
2nd. Every lag from that point on is off by a day, quietly, for the rest of the series. Nothing
raises; the feature is a real number of rentals from a real day, just the wrong one.

That is **the most common bug in time-series feature engineering**, and it is created by a missing
row rather than by any code you wrote. Module 09 opens with it.

**What resampling gives you** is a *complete* index: one row per period, whether or not the data
had one. Missing periods become explicit, and then you have to make an explicit decision about
them - which is the point.

In [ ]:
print("as a sum, the gap becomes 0 - which claims the stand was open and rented nothing:")
print(per_day.resample("D").sum().to_string())
print("\nas an aggregation of nothing, it becomes NaN - which says 'we do not know':")
print(per_day.resample("D").mean().to_string())
print("\nasfreq keeps the values and shows the gap honestly:")
print(per_day.asfreq("D").to_string())

### Diagnosis

**`resample("D").sum()` filled the gap with `0`.** That is a *claim*: it says the stand was open on
March 3rd and rented nothing. If the stand was closed, or the sensor was broken, or the export
dropped a row, then `0` is a fabricated observation, and any model trained on it learns that
Sundays are dead.

**`.mean()` and `.asfreq()` give `NaN`**, which says the honest thing: *there is no measurement
here.*

**The decision is yours and it is not a technical one:**

| The gap means | Correct fill | Why |
|---|---|---|
| Genuinely zero activity - open, nobody came | `0` | It is a real measurement |
| Closed, broken sensor, failed export | `NaN` | It is an absence of measurement, and pretending otherwise invents data |
| You do not know which | `NaN`, and go and find out | The one thing you must not do is guess silently |

The three cases are indistinguishable from the data alone. Telling them apart requires knowing how
the data was produced - which is chapter 02-02, and is why that chapter exists.

### Remedies

| Remedy | What it catches | Cost |
|---|---|---|
| `resample` / `asfreq` instead of grouping by date | Missing periods, permanently | Nothing - it is the same length of line |
| Compare `len(series)` with the number of days in the range | Gaps you did not know about | One line |
| Decide `0` versus `NaN` deliberately, and write down why | Fabricated observations | A sentence in the notebook |
| Build lags on a complete index only | Every off-by-one lag bug downstream | Free, if the index is complete |

## Common misconceptions

**"`groupby` sorts my data, so the result is in a random order."**
It sorts by the group key, deterministically. Pass `sort=False` to keep first-appearance order,
which is faster on large data and sometimes what you want for reporting.

**"A left join can't lose rows, so it's safe."**
It cannot lose rows; it can *multiply* them. Losing rows is loud - your count drops. Multiplying is
quiet, and worse.

**"`NaN` rows are dropped by `groupby`, which is fine."**
They are dropped by default (`dropna=True`), and it is fine only if you know. If 20% of your rows
have a missing key, a `groupby` total silently excludes them and does not sum to the overall total.
Check that your group totals add up.

**"Resampling and grouping by date are the same thing."**
They differ exactly on the periods with no data - which is the case you care about.

**"`.apply` is the flexible way to do anything per group."**
It is the slow way, and its return shape depends on what your function returns, which makes it
unpredictable. Use `agg` when you want one row per group and `transform` when you want the original
shape. Reach for `.apply` only when neither fits.

**"Time zones are a detail I can deal with later."**
A timestamp with no timezone is ambiguous, and mixing aware and naive timestamps raises in some
operations and silently misaligns in others. Decide early: store UTC, convert for display. Daylight
saving alone makes "group by hour" a genuinely hard question twice a year.

---

## Exercises

Solutions: `solutions/01_python_bridge/01-05_pandas_group_join_time_solutions.ipynb`.

### Quick understanding

**E1 (define).** What are the three steps `groupby` performs, and what is the difference between
`agg` and `transform`?

**E2 (explain).** Why is a left join that *multiplies* rows more dangerous than an inner join that
*loses* them?

**E3 (explain).** Why does `resample("D")` produce a row for March 3rd when `groupby(.dt.date)`
does not, and when is that difference a bug?

### Hand calculation

**E4 (calculate).** Left frame: 5 rows with keys `[a, a, b, c, d]`. Right frame: keys
`[a, b, b, e]`. Without running anything, give the number of rows for a left join, an inner join
and an outer join, and say which keys produce unmatched rows.

**E5 (calculate).** Station a has counts 120, 150, 90, 200 on the 1st, 2nd, 4th and 5th. Compute by
hand: the mean, the day-on-day differences as the data is stored, and the day-on-day differences if
the 3rd is inserted as a missing value. Which difference series would you use as a feature, and
why?

### Coding

**E6 (code).** Produce a table with one row per site containing: number of station-days, total
rentals, mean rentals per station-day, and rentals per unit of capacity. Use a join, a `groupby`
and named aggregations.

**E7 (code).** Write `safe_merge(left, right, on, how="left", expect="many_to_one")` that performs
the merge, asserts the row count behaved as expected for that join type, reports how many rows
found no match, and returns the result. Demonstrate it passing on `stations` and failing on
`stations_messy`.

### Interpretation

**E8 (interpret).** The inflated join changed the total from 1125 to 1480, but the maximum count
stayed at 200 and the mean moved only from 93.75 to 92.50. Say which statistics duplication
distorts and which it leaves alone, and explain why "the mean barely moved" is not reassurance.

### Debugging

**E9 (diagnose).** A daily forecasting model was doing well and got much worse after the data team
"fixed a gap by filling missing days with zero". Explain what probably happened, name two ways to
confirm it, and say what the right fix would have been.

### Exam and interview reasoning

**E10 (defend).** *"Why do you check the row count on every join? Surely you know your own data."*
Answer in four sentences.

**E11 (design).** You are joining a transactions table (10 million rows) to a customer table where
you are *not* sure the customer id is unique. Describe the checks you would run first, in order,
and what you would do if the ids turn out not to be unique.

### Transfer to a different situation

**E12 (design).** A hospital has an admissions table (one row per admission) and a lab-results
table (many rows per admission). You want one row per admission with the patient's *worst* lab
value. Say which operation goes first and why, what the row count should be at each step, and
what would go wrong if you did it in the other order.

### Explain it to someone non-technical

**E13 (explain).** In under 70 words, explain to Maria why filling the missing day with a zero made
the forecast worse, when a zero seems like the obvious thing to put there.

### Optional challenge

**E14 (code + diagnose).** Build a `lag_1` feature - yesterday's rentals for each station - two
ways: once on the data as stored, and once on a complete daily index per station. Show that they
differ, identify exactly which rows differ, and quantify how a model would be misled. Then say how
you would build this feature so the bug cannot happen.

In [ ]:
# Your workspace. Still in memory: rentals, stations, stations_messy, joined, inflated,
# daily, per_day, resampled, grouped.

## Mastery check

Without scrolling up, can you:

- [ ] Say what `groupby` does in three steps, and when to use `transform`? *(If not: "groupby".)*
- [ ] Name the four join types and what each is for? *(If not: "Joining".)*
- [ ] Say the two lines that make a join safe? *(If not: "Failure lab 1".)*
- [ ] Explain why a duplicated key inflates a model's score? *(If not: "Failure lab 1 diagnosis".)*
- [ ] Say the difference between `resample("D").sum()` and `groupby(.dt.date).sum()` on a series
      with a gap? *(If not: "Failure lab 2".)*

## What should now feel instinctive

1. **Count the rows before and after every join**, and pass `validate=`.
2. **A duplicated key is a data problem**, not a join problem - find the row, do not just change
   the join.
3. **`agg` collapses, `transform` broadcasts.** Pick the one that matches the shape you want.
4. **Resample rather than group by date** whenever the index is time, so gaps stay visible.
5. **`0` and `NaN` are different claims.** One says "we measured nothing happening", the other says
   "we did not measure". Choose deliberately.

## Flashcards

| Question | Answer |
|---|---|
| The three steps of `groupby` | Split by key, apply a function, combine the results |
| `agg` vs `transform` | One row per group vs one row per original row, with the group value broadcast back |
| The four join types | left, inner, outer, right - keep the left, only matches, everything, the right |
| The one thing to check on every join | The row count, before and after |
| `validate="many_to_one"` | Raises if the right-hand keys are not unique - turns silent multiplication into an error |
| `indicator=True` | Adds a `_merge` column saying which side each row matched |
| Why does a duplicated key inflate a model's score? | It copies observations, which then land on both sides of a split - duplicate leakage |
| `resample("D")` vs `groupby(.dt.date)` | Resample produces a row for every period, including empty ones; grouping omits them |
| Why does a missing day break a lag feature? | "The previous row" is no longer "the previous day", so every lag after the gap is wrong |
| `0` vs `NaN` for a gap | Zero is a measurement of nothing happening; NaN is the absence of a measurement |

## Next

**01-06 · Plotting that says something, and reproducible randomness** - the last chapter of the
optional module.

You can now load, clean, group, join and index by time. The two remaining tools are the ones that
make results *believable*: a chart with labelled axes and units that does not mislead, and a seeded
random number generator so that the number you report today is the number you get tomorrow.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).